State-of-the art methods


Importation des données en forme de DataFrame pandas

In [ ]:
import pandas as pd
import numpy as np
import os

# Liste vide pour stocker les lignes
data = []

# Exemple de boucle sur les fichiers
for subject_id in range(1, 11):  # 10 sujets
    for digit in range(10):      # 0 à 9
        for repetition in range(1, 11):  # 10 répétitions de 1 à 10
            # Ex: "subject1_digit2_rep5.txt"
            filename = f"Subject{subject_id}-{digit}-{repetition}.csv"
            filepath = os.path.join("Domain1_csv", filename)

            if os.path.exists(filepath):
                # Charger uniquement x, y, z (on ignore t)
                points = np.loadtxt(filepath, delimiter=",", skiprows=1, usecols=(0,1,2))
                
                data.append({
                    "subject_id": subject_id,
                    "digit": digit,
                    "repetition": repetition,
                    "points": points
                })
            else:
                print("Fichier manquant :", filepath)    

# Convertir en DataFrame
df = pd.DataFrame(data)



Centrage et normalisation des points

In [ ]:
def center_and_normalize(gesture):
    center = np.mean(gesture, axis=0) # Calcule moyenne par colonne x,y,z
    centered = gesture - center
    norm = np.linalg.norm(centered)  # Cacul de la norme des points
    return centered / norm

# Appliquer la fonction center_and_normalize --> Ajoute une colonne points_normalized 
df["points_normalized"] = df["points"].apply(center_and_normalize)


Méthode : SVD (pour réduction de dimension) pour transformer tes gestes en vecteurs compacts, puis les envoyer dans un classifieur comme un SVM ou une régression.

In [ ]:
# Interpoler les données 
from scipy.interpolate import interp1d
import numpy as np

def interpolate_gesture(points, target_length=100):
    # points : (N, 3)
    n_points = len(points)
    if n_points == 0:
        return np.zeros((target_length, 3))

    # Positions normalisées entre 0 et 1
    original_t = np.linspace(0, 1, n_points)
    target_t = np.linspace(0, 1, target_length)

    # Interpolation sur x, y, z séparément
    interp_x = interp1d(original_t, points[:, 0], kind='linear')
    interp_y = interp1d(original_t, points[:, 1], kind='linear')
    interp_z = interp1d(original_t, points[:, 2], kind='linear')

    interpolated = np.stack([
        interp_x(target_t),
        interp_y(target_t),
        interp_z(target_t)
    ], axis=1)

    return interpolated


In [ ]:
# Appliquer l'interpolation
df["interpolated"] = df["points_normalized"].apply(lambda x: interpolate_gesture(x, target_length=100))


In [ ]:
#Vérification de l'interpolation
df["len_normalized"] = df["points_normalized"].apply(len)
df["len_interpolated"] = df["interpolated"].apply(len)

# Afficher quelques exemples
print(df[["subject_id", "digit", "len_normalized", "len_interpolated"]].head(10))  

In [ ]:
# Transforme en vecteur 1D
df["flattened"] = df["interpolated"].apply(lambda x: x.flatten())

# Rassemble tous les vecteurs aplati x en un seul tableau X
X = np.stack(df["flattened"].values)  # shape : (nb_gestes, 300)
y = df["digit"].values                # labels 0 à 9
subject_ids = df["subject_id"].values

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=50,random_state=42)  # Réduis le vecteur
X_reduced = svd.fit_transform(X)

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

def leave_one_user_out_vector_model(X, y, subject_ids, model, output_csv):
    
    accuracies = []
    results = []

    unique_users = np.unique(subject_ids)

    for user in unique_users:
        # Séparer train/test selon l'utilisateur
        test_idx = (subject_ids == user)
        train_idx = ~test_idx

        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        accuracies.append(acc)
        results.append({"user": int(user), "accuracy": acc})
        print(f"User {user} – Accuracy : {acc:.3f}")

    # Moyenne et écart-type
    avg = np.mean(accuracies)
    std = np.std(accuracies)

    print(f"\nMoyenne globale : {avg:.3f} | Écart-type : {std:.3f}")
    results.append({"user": "average", "accuracy": avg})
    results.append({"user": "std", "accuracy": std})

    # Sauvegarde dans un CSV
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv, index=False)
    print(f"Résultats enregistrés dans {output_csv}")

    return df_results


In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

def user_dependent_vector_model(X, y, subject_ids, digits, model, output_csv):
    
    results = []
    users = np.unique(subject_ids)

    for user in users:
        accs = []
        user_idx = np.where(subject_ids == user)[0]
        user_X = X[user_idx]
        user_y = y[user_idx]
        user_digits = digits[user_idx]

        for fold in range(10):
            train_idx, test_idx = [], []

            # Pour chaque chiffre (0 à 9), on garde 1 exemple pour le test
            for digit in range(10):
                digit_indices = np.where(user_digits == digit)[0]
                if len(digit_indices) < 10:
                    continue  # sécurité

                np.random.seed(fold)
                np.random.shuffle(digit_indices)

                test_digit_idx = digit_indices[fold % len(digit_indices)]
                test_idx.append(test_digit_idx)

                train_digit_idx = [i for i in digit_indices if i != test_digit_idx]
                train_idx.extend(train_digit_idx)

            # Indices globaux
            test_global_idx = user_idx[test_idx]
            train_global_idx = user_idx[train_idx]

            X_train, y_train = X[train_global_idx], y[train_global_idx]
            X_test, y_test = X[test_global_idx], y[test_global_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            acc = accuracy_score(y_test, y_pred)
            accs.append(acc)

        mean_acc = np.mean(accs)
        std_acc = np.std(accs)
        print(f"User {user} – Moyenne : {mean_acc:.3f} | Écart-type : {std_acc:.3f}")
        results.append({"user": user, "mean_accuracy": mean_acc, "std_accuracy": std_acc})

    # Moyenne globale
    global_mean = np.mean([r["mean_accuracy"] for r in results])
    global_std = np.std([r["mean_accuracy"] for r in results])

    print(f"\nMoyenne globale : {global_mean:.3f} | Écart-type : {global_std:.3f}")
    results.append({"user": "average", "mean_accuracy": global_mean, "std_accuracy": None})
    results.append({"user": "std", "mean_accuracy": global_std, "std_accuracy": None})

    df_results = pd.DataFrame(results)
    df_results.to_csv(output_csv, index=False)
    print(f"Résultats enregistrés dans {output_csv}")

    return df_results


LogisticRegression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
results=leave_one_user_out_vector_model(X_reduced, y, subject_ids, model,output_csv="user_independent_results_svd.csv")

results=user_dependent_vector_model(X_reduced, y, subject_ids, df["digit"].values, model,output_csv="user_dependent_results_svd.csv")

In [ ]:
from sklearn.svm import SVC

# Modèle SVM linéaire
svm_model = SVC(kernel='linear', C=10)  #  C curseur entre complexité et robustesse

#Validation croisée user independant
results=leave_one_user_out_vector_model(X_reduced,y,subject_ids,model=svm_model, output_csv="user_independent_svm.csv")

# Validation croisée user dependent
results = user_dependent_vector_model(X_reduced,y,subject_ids,df["digit"].values,model=svm_model, output_csv="user_dependent_svm.csv")

Réseaux de neurones MLP

In [ ]:
from sklearn.neural_network import MLPClassifier
import warnings
from sklearn.exceptions import ConvergenceWarning

# Supprimer les gros messages
warnings.filterwarnings("ignore", category=ConvergenceWarning)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(100,),   # 1 couche cachée de 100 neurones
    activation='relu',           # fonction d’activation
    solver='adam',               # optimiseur
    alpha=0.001,                # régularisation L2
    learning_rate='adaptive',    # learning rate qui s’adapte
    max_iter=1000,                # nombre d’itérations max
    random_state=42
)

In [ ]:
#Validation croisée user independant
results=leave_one_user_out_vector_model(X_reduced,y,subject_ids,model=mlp_model,output_csv="user_dependent_mlp.csv")

# Validation croisée user dependent
results=user_dependent_vector_model(
    X_reduced,                # vecteurs SVD
    y,                        # labels (digits)
    subject_ids,              # identifiants utilisateur
    df["digit"].values,       # pour regrouper les digits dans les folds
    model=mlp_model,
    output_csv="user_dependent_mpl.csv"
)